# Exploratory Data Analysis & Cleaning

This notebook analyzes the dataset column by column to identify preprocessing requirements and applies the corresponding cleaning steps using the preprocessing pipeline.

## Imports

In [319]:
import numpy as np

from src.preprocessing.cleaner import *
from src.preprocessing.config import INPUT_DIR
from src.preprocessing.loader import load_data

## Pandas Settings

In [320]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", 2500)
pd.set_option("display.max_colwidth", None)

## Load Dataset

In [321]:
df = load_data(INPUT_DIR)

# Preview of Dataset

In [322]:
# --------------- Prints first 10 rows of dataset ---------------
print(df.head(10))

  Invoice StockCode                          Description  Quantity          InvoiceDate  Price  Customer ID         Country
0  489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12  2009-12-01 07:45:00   6.95      13085.0  United Kingdom
1  489434    79323P                   PINK CHERRY LIGHTS        12  2009-12-01 07:45:00   6.75      13085.0  United Kingdom
2  489434    79323W                  WHITE CHERRY LIGHTS        12  2009-12-01 07:45:00   6.75      13085.0  United Kingdom
3  489434     22041         RECORD FRAME 7" SINGLE SIZE         48  2009-12-01 07:45:00   2.10      13085.0  United Kingdom
4  489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24  2009-12-01 07:45:00   1.25      13085.0  United Kingdom
5  489434     22064           PINK DOUGHNUT TRINKET POT         24  2009-12-01 07:45:00   1.65      13085.0  United Kingdom
6  489434     21871                  SAVE THE PLANET MUG        24  2009-12-01 07:45:00   1.25      13085.0  United Kingdom
7  48943

# Dataset Basic Information

**Cleaning:** Standardize column names, and columns with data type `str` except InvoiceDate.

The dataset initially contains **525,461 rows** and **8 columns**. It has **110,855 missing values** and **6,865 duplicate rows**.

The columns are:

```text
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
 'Price', 'Customer ID', 'Country']
```

The dataset contains **28,816 unique transactions**, including completed transactions, cancellations, and other invalid or non-product transactions.


In [323]:
print("Shape of dataframe: ", df.shape)
print("Number of Missing Values: ", df.isnull().sum().sum())
print("Duplicate Values: ", df.duplicated().sum())
print("Column Names: ", df.columns)
print("Number of Transactions: ", df["Invoice"].nunique())
print(f"Data types:\n{df.dtypes}")

Shape of dataframe:  (525461, 8)
Number of Missing Values:  110855
Duplicate Values:  6865
Column Names:  Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country'], dtype='str')
Number of Transactions:  28816
Data types:
Invoice            str
StockCode          str
Description        str
Quantity         int64
InvoiceDate        str
Price          float64
Customer ID    float64
Country            str
dtype: object


## Cleaning

In [324]:
df = standardize(
    df,
    standardize_columns=True,
    columns=["Invoice", "StockCode", "Description", "Country"],
)
print(f"Standardized column names: {df.columns}")

Standardized column names: Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer_ID', 'Country'], dtype='str')


# Customer ID Analysis
### Missing Customer ID

20.54% of transactions have missing `Customer_ID`. Since the project operates at the customer level, these transactions cannot be reliably associated with a customer and are therefore removed.

## Basic Information

In [325]:
print(f"Customer ID data type: {df["Customer_ID"].dtypes}")
print(f"Number of missing values: {df['Customer_ID'].isnull().sum()}")
print(
    f"Percentage of missing Customer IDs: {round(df['Customer_ID'].isnull().mean() * 100, 2)}%"
)
print(f"Number of unique values: {df['Customer_ID'].nunique()}")
print(f"Range: {df['Customer_ID'].max() - df['Customer_ID'].min()}")

Customer ID data type: float64
Number of missing values: 107927
Percentage of missing Customer IDs: 20.54%
Number of unique values: 4383
Range: 5941.0


## Sample Values

In [326]:
print(df["Customer_ID"].sample(20))

2062          NaN
65068     14293.0
46633     15768.0
46204     13338.0
39445     14646.0
43859     14063.0
463573    15850.0
516063    17218.0
219274    14136.0
433198    13137.0
432487    15379.0
284112        NaN
145940    16954.0
116439        NaN
390168    16130.0
27664     16409.0
228133    12840.0
367431        NaN
403180    13373.0
505071    12921.0
Name: Customer_ID, dtype: float64


## Missing Values Inspection

In [327]:
print(df[df["Customer_ID"].isna()].describe())

            Quantity          Price  Customer_ID
count  107927.000000  107927.000000          0.0
mean        0.971045       7.788750          NaN
std       128.207415     290.474071          NaN
min     -9600.000000  -53594.360000          NaN
25%         1.000000       1.660000          NaN
50%         1.000000       3.360000          NaN
75%         2.000000       5.910000          NaN
max     10200.000000   25111.090000          NaN


## Cleaning

In [328]:
df = clean_customer_id(df)
print(f"Missing values after cleaning: {df["Customer_ID"].isnull().sum()}")

Missing values after cleaning: 0


# Invoice Analysis

`Invoice` contains two types of values:

* **Numeric values** represent completed transactions, e.g. `486001`.
* **Values starting with `C`** represent cancelled transactions, e.g. `C534826`.

Cancelled transactions are removed because they do not represent completed customer purchases and are not relevant to repeat-purchase behavior.


## Basic Information

In [329]:
print(f"Data type: {df["Invoice"].dtypes}")
print(f"Number of missing values: {df["Invoice"].isnull().sum()}")
print(f"Number of unique values: {df['Invoice'].nunique()}")
print(
    f"Number of cancelled transactions: {df.loc[df["Invoice"].str.startswith("C"), "Invoice"].nunique()}"
)

Data type: str
Number of missing values: 0
Number of unique values: 23587
Number of cancelled transactions: 4372


## Sample Values

In [330]:
print(df["Invoice"].sample(20))

399070     527505
206410     509350
265415     515016
133300     502055
369773     525084
471489     533798
100430     498890
397928     527439
107616     499752
182401     506709
285488     517125
492919     535457
179292     506391
351992     523638
316887     520218
462433     533069
326055     521196
486256    C534845
7046       490013
498367     535974
Name: Invoice, dtype: str


## Unusual Invoice Values

In [331]:
print(
    f"Number of unusual values: {df[df["Invoice"].str.contains(r"\D", regex=True)]["Invoice"].nunique()}"
)
print(
    sorted(df[df["Invoice"].str.contains(r"\D", regex=True)]["Invoice"].unique())[:10]
)
print(
    sorted(df[df["Invoice"].str.contains(r"\D", regex=True)]["Invoice"].unique())[-10:]
)

Number of unusual values: 4372
['C489449', 'C489459', 'C489476', 'C489503', 'C489504', 'C489518', 'C489524', 'C489527', 'C489528', 'C489530']
['C538112', 'C538114', 'C538115', 'C538118', 'C538119', 'C538121', 'C538122', 'C538123', 'C538124', 'C538164']


## Cleaning

In [332]:
df = clean_invoice(df)

# StockCode Analysis

### Bank Charges

Rows with `StockCode = "BANK CHARGES"` are removed because they represent bank charges rather than customer product purchases. Since this project focuses on predicting repeat product purchases, these transactions are not relevant to the target behavior and could distort customer-level purchasing features.

### Non-Product and Adjustment Codes

`StockCode` values such as `POST`, `D`, `M`, `C2`, `PADS`, and `ADJUST` represent non-product transactions or adjustment codes rather than regular products. These values are removed because they are not relevant to customer product-purchase behavior.

### Invalid StockCodes

`StockCodes` such as `TEST001`, `TEST002`, `ADJUST2`, and `SP1002` do not represent regular product codes. These values are removed because they are associated with test or adjustment-related transactions rather than actual products.

## Basic Information

In [333]:
print(f"Data type: {df["StockCode"].dtypes}")
print(f"Number of missing values: {df["StockCode"].isnull().sum()}")
print(f"Number of unique values: {df["StockCode"].nunique()}")

Data type: str
Number of missing values: 0
Number of unique values: 4017


## Sample Values

In [334]:
print(df["StockCode"].sample(20))

79596      21700
104443    84997D
171567     21179
48221      84077
111239    84510A
53418      21947
237327    85232B
8839       21752
285926     22047
514622     22834
216017     22099
238907     22211
517606     22577
414282     22300
110199     21745
315666     21070
515779     22659
196155    84596J
15089      84978
423475     21832
Name: StockCode, dtype: str


## Unusual Values

In [335]:
# --------------- StockCodes containing non-alphanumeric characters ---------------
print(df.loc[~df["StockCode"].str.fullmatch(r"[A-Za-z0-9]+")]["StockCode"].unique())

<StringArray>
['BANK CHARGES']
Length: 1, dtype: str


In [336]:
# --------------- StockCode == BANK CHARGES ---------------
print(df[df["StockCode"] == "BANK CHARGES"])

       Invoice     StockCode   Description  Quantity          InvoiceDate  Price  Customer_ID         Country
18466   490948  BANK CHARGES  BANK CHARGES         1  2009-12-08 14:29:00   15.0      16805.0  UNITED KINGDOM
94431   498269  BANK CHARGES  BANK CHARGES         1  2010-02-17 15:03:00   15.0      16928.0  UNITED KINGDOM
148098  503497  BANK CHARGES  BANK CHARGES         1  2010-04-01 12:07:00   15.0      17539.0  UNITED KINGDOM
153573  503960  BANK CHARGES  BANK CHARGES         1  2010-04-08 16:50:00   15.0      12843.0  UNITED KINGDOM
167424  505204  BANK CHARGES  BANK CHARGES         1  2010-04-20 16:24:00   15.0      17448.0  UNITED KINGDOM
206572  509375  BANK CHARGES  BANK CHARGES         1  2010-05-21 14:40:00   15.0      17448.0  UNITED KINGDOM
210149  509669  BANK CHARGES  BANK CHARGES         1  2010-05-25 12:03:00   15.0      17448.0  UNITED KINGDOM
231102  511774  BANK CHARGES  BANK CHARGES         1  2010-06-10 12:16:00   15.0      17032.0  UNITED KINGDOM
240612  51

In [337]:
# --------------- Values with unusual length ---------------
print(df[df["StockCode"].str.len() < 5]["StockCode"].unique())

<StringArray>
['POST', 'C2', 'M', 'PADS', 'D']
Length: 5, dtype: str


In [338]:
# --------------- Analysis of StockCodes with length less than 5 ---------------
for value in ["POST", "D", "M", "C2", "PADS"]:
    print(df[df["StockCode"] == value].head())

     Invoice StockCode Description  Quantity          InvoiceDate  Price  Customer_ID  Country
89    489439      POST     POSTAGE         3  2009-12-01 09:28:00   18.0      12682.0   FRANCE
126   489444      POST     POSTAGE         1  2009-12-01 09:55:00  141.0      12636.0      USA
173   489447      POST     POSTAGE         1  2009-12-01 10:10:00  130.0      12362.0  BELGIUM
625   489526      POST     POSTAGE         6  2009-12-01 11:50:00   18.0      12533.0  GERMANY
1244  489557      POST     POSTAGE         4  2009-12-01 12:52:00   18.0      12490.0   FRANCE
       Invoice StockCode Description  Quantity          InvoiceDate   Price  Customer_ID         Country
160443  504700         D    DISCOUNT         1  2010-04-15 18:08:00   57.63      17032.0  UNITED KINGDOM
212633  509979         D    DISCOUNT         1  2010-05-26 14:07:00  101.99      12843.0  UNITED KINGDOM
312285  519808         D    DISCOUNT       192  2010-08-20 12:50:00    1.00      16422.0  UNITED KINGDOM
494226  53

In [339]:
# --------------- Analysis of StockCodes which doesn't contains numbers ---------------
print(df.loc[~df["StockCode"].str.contains(r"\d")]["StockCode"].unique())

<StringArray>
['POST', 'M', 'BANK CHARGES', 'PADS', 'ADJUST', 'D']
Length: 6, dtype: str


In [340]:
# --------------- StockCode == ADJUST ---------------
print(df[df["StockCode"] == "ADJUST"].head(10))

      Invoice StockCode                          Description  Quantity          InvoiceDate   Price  Customer_ID   Country
70976  495733    ADJUST  ADJUSTMENT BY JOHN ON 26/01/2010 16         1  2010-01-26 16:21:00   68.34      14911.0      EIRE
70977  495735    ADJUST  ADJUSTMENT BY JOHN ON 26/01/2010 16         1  2010-01-26 16:22:00  201.56      12745.0      EIRE
70978  495734    ADJUST  ADJUSTMENT BY JOHN ON 26/01/2010 16         1  2010-01-26 16:22:00  205.82      14911.0      EIRE
70980  495736    ADJUST  ADJUSTMENT BY JOHN ON 26/01/2010 16         1  2010-01-26 16:23:00   21.00      12606.0     SPAIN
70985  495742    ADJUST  ADJUSTMENT BY JOHN ON 26/01/2010 16         1  2010-01-26 16:25:00   63.24      12404.0   FINLAND
71022  495745    ADJUST  ADJUSTMENT BY JOHN ON 26/01/2010 16         1  2010-01-26 16:26:00   56.73      12466.0    FRANCE
71023  495748    ADJUST  ADJUSTMENT BY JOHN ON 26/01/2010 16         1  2010-01-26 16:26:00  117.72      16291.0  PORTUGAL
71033  495747   

In [341]:
# --------------- Values starting with anything but a digit ---------------
df.loc[~df["StockCode"].str.match(r"^\d", na=False), "StockCode"].unique()

<StringArray>
['POST', 'C2', 'M', 'BANK CHARGES', 'TEST001', 'TEST002', 'PADS', 'ADJUST', 'D', 'ADJUST2', 'SP1002']
Length: 11, dtype: str

## StockCode and Description Relationship

In [342]:
print(df.groupby("StockCode")["Description"].nunique().value_counts().sort_index())

Description
1    3617
2     378
3      17
4       5
Name: count, dtype: int64


In [343]:
print(
    df[
        df["StockCode"].isin(
            df.groupby("StockCode")["Description"].nunique().loc[lambda x: x == 4].index
        )
    ]["Description"].unique()
)

<StringArray>
['RED SPOTTY COIR DOORMAT', 'PARTY PIZZA DISH PINK+WHITE SPOT', 'PARTY PIZZA DISH GREEN+WHITE SPOT', 'PARTY PIZZA DISH BLUE+WHITE SPOT', 'PARTY PIZZA DISH BLUE WHITE SPOT', 'PARTY PIZZA DISH PINK WHITE SPOT', 'PARTY PIZZA DISH GREEN WHITE SPOT', 'LUNCHBAG PINK RETROSPOT', 'LUNCH BAG PINK RETROSPOT', 'DOOR MAT RED SPOT', 'PARTY PIZZA DISH BLUE RETROSPOT', 'PARTY PIZZA DISH PINK RETROSPOT', 'PARTY PIZZA DISH GREEN RETROSPOT', 'DOORMAT RED SPOT', 'LUNCH BAG PINK POLKADOT', 'DOORMAT RED RETROSPOT', 'LUNCH BAG PINK POLKADOTS', 'PARTY PIZZA DISH PINK POLKADOT', 'PARTY PIZZA DISH BLUE POLKADOT', 'PARTY PIZZA DISH GREEN POLKADOT']
Length: 20, dtype: str


## Cleaning

In [344]:
df = clean_stock_code(df)

# Description Analysis

### Multiple Descriptions per StockCode

Some `StockCode` values have multiple descriptions. These are considered valid for now, as the differences may represent variations in product descriptions rather than invalid transactions. No rows are removed based on this condition.


## Basic Information

In [345]:
print(f"Data type: {df["Description"].dtypes}")
print(f"Number of missing values: {df["Description"].isnull().sum()}")
print(f"Number of unique values: {df["Description"].nunique()}")
print(
    f"Average description per stock code: {round(df["Description"].nunique() / df["StockCode"].nunique(), 2)}"
)

Data type: str
Number of missing values: 0
Number of unique values: 4390
Average description per stock code: 1.1


## Sample Values

In [346]:
print(df["Description"].sample(20))

500877                HAND WARMER UNION JACK
57656        RED SPOT HEART HOT WATER BOTTLE
334529          PACK OF 6 HANDBAG GIFT BOXES
153970          200 RED + WHITE BENDY STRAWS
126042             RETRO SPOT SMALL MILK JUG
207108                      PINK SPOTTY BOWL
517758        BLACK TEA TOWEL CLASSIC DESIGN
144213       VINTAGE UNION JACK SHOPPING BAG
482100              HOME BUILDING BLOCK WORD
508203          CHRISTMAS LIGHTS 10 REINDEER
29035                 GLITTER CHRISTMAS STAR
160140              LUNCH BAG PINK RETROSPOT
341792       HANGING HEART MIRROR DECORATION
90034                          POTTERING MUG
188957                  LUNCH BAG RED SPOTTY
126311       SET 12 KIDS COLOUR CHALK STICKS
307798                       RED SPOTTY BOWL
105399                       DOOR MAT HEARTS
272718    WHITE HANGING HEART T-LIGHT HOLDER
490712                       DOORMAT TOPIARY
Name: Description, dtype: str


## Description and StockCode Relationship

In [347]:
df.groupby("StockCode")["Description"].nunique().loc[lambda x: x > 1].sort_values(
    ascending=False
)

StockCode
22345     4
22346     4
22384     4
22344     4
20685     4
21523     3
21524     3
21955     3
22356     3
22191     3
22139     3
22333     3
22343     3
22536     3
22740     3
22844     3
22845     3
22852     3
22853     3
22952     3
84509C    3
85099B    3
84509F    2
21041     2
84969     2
84968F    2
20682     2
17129F    2
84509G    2
84510C    2
84899E    2
84905     2
84907     2
84918     2
84919     2
84951A    2
84951B    2
84968A    2
84968B    2
84968C    2
84968D    2
84968E    2
15058B    2
21042     2
21080     2
21121     2
21122     2
21123     2
21124     2
21154     2
21155     2
21156     2
21157     2
21210     2
21212     2
21216     2
21217     2
21238     2
21239     2
21240     2
21241     2
21242     2
21243     2
21244     2
21245     2
21246     2
21249     2
21284     2
21285     2
21286     2
21291     2
21391     2
21392     2
21393     2
21394     2
21395     2
21397     2
21398     2
21399     2
21485     2
21495     2
21498     2
21499 

In [348]:
print(
    df.groupby("StockCode")["Description"]
    .agg(["nunique", lambda x: x.unique().tolist()])
    .query("nunique > 1")
    .sort_values("nunique", ascending=False)
)

           nunique                                                                                                                                 <lambda_0>
StockCode                                                                                                                                                    
22345            4      [PARTY PIZZA DISH BLUE+WHITE SPOT, PARTY PIZZA DISH BLUE WHITE SPOT, PARTY PIZZA DISH BLUE RETROSPOT, PARTY PIZZA DISH BLUE POLKADOT]
22346            4  [PARTY PIZZA DISH GREEN+WHITE SPOT, PARTY PIZZA DISH GREEN WHITE SPOT, PARTY PIZZA DISH GREEN RETROSPOT, PARTY PIZZA DISH GREEN POLKADOT]
22384            4                                     [LUNCHBAG PINK RETROSPOT, LUNCH BAG PINK RETROSPOT, LUNCH BAG PINK POLKADOT, LUNCH BAG PINK POLKADOTS]
22344            4      [PARTY PIZZA DISH PINK+WHITE SPOT, PARTY PIZZA DISH PINK WHITE SPOT, PARTY PIZZA DISH PINK RETROSPOT, PARTY PIZZA DISH PINK POLKADOT]
20685            4                                  

# Quantity Analysis

**Cleaning:** `Negative` quantities were caused by cancelled transactions and became zero after cancelled invoices were removed. Extremely `large` positive quantities were investigated and appear to represent legitimate bulk purchases, so they are retained.

## Basic Information

In [349]:
print(f"Data type: {df["Quantity"].dtypes}")
print(f"Number of missing values: {df["Quantity"].isnull().sum()}")
print(f"Number of unique values: {df["Quantity"].nunique()}")
print(f"Number of duplicate values: {df['Quantity'].duplicated().sum()}")

Data type: int64
Number of missing values: 0
Number of unique values: 343
Number of duplicate values: 405980


## Statistics

In [350]:
print(f"Median: {np.median(df['Quantity'])}")
print(df["Quantity"].describe())

Median: 5.0
count    406323.000000
mean         13.619534
std          97.002302
min           1.000000
25%           2.000000
50%           5.000000
75%          12.000000
max       19152.000000
Name: Quantity, dtype: float64


## Negative or Zero Values

In [351]:
print(
    f"Number of Negative or Zero values: {df[df["Quantity"] <= 0].value_counts().sum()}"
)

Number of Negative or Zero values: 0


## Extreme Quantity Values Analysis

In [352]:
print(
    f"Number of extreme values (> 1000) in Quantity are: {df[df["Quantity"] > 1000].value_counts().sum()}\n"
)
print(df[df["Quantity"] > 1000])

Number of extreme values (> 1000) in Quantity are: 199

       Invoice StockCode                          Description  Quantity          InvoiceDate  Price  Customer_ID         Country
7302    490018     21981          PACK OF 12 WOODLAND TISSUES      4320  2009-12-03 12:31:00   0.25      17940.0  UNITED KINGDOM
7303    490018     21967             PACK OF 12 SKULL TISSUES      5184  2009-12-03 12:31:00   0.25      17940.0  UNITED KINGDOM
7304    490018     21984      PACK OF 12 PINK PAISLEY TISSUES      4008  2009-12-03 12:31:00   0.25      17940.0  UNITED KINGDOM
7305    490018     21980        PACK OF 12 RED SPOTTY TISSUES      4008  2009-12-03 12:31:00   0.25      17940.0  UNITED KINGDOM
22100   491133     16014          SMALL CHINESE STYLE SCISSOR      1500  2009-12-09 16:17:00   0.32      13848.0  UNITED KINGDOM
23689   491212     20668      DISCO BALL CHRISTMAS DECORATION      1002  2009-12-10 14:28:00   0.10      14062.0  UNITED KINGDOM
24815   491440     72756                 

In [353]:
print(df[df["Quantity"] > 1000].describe()["Price"])

count    199.000000
mean       0.379799
std        0.495585
min        0.030000
25%        0.130000
50%        0.180000
75%        0.360000
max        2.750000
Name: Price, dtype: float64


In [354]:
print(df["Quantity"].value_counts().sort_index().tail(10))

Quantity
7008     2
7128     3
9312     1
9360     1
9456     1
10000    4
12480    1
12744    1
12960    2
19152    1
Name: count, dtype: int64


In [355]:
extreme_values = [19152, 12960, 12744, 12480, 10000, 9456, 9360, 9312, 7128, 7008]
print(df[df["Quantity"].isin(extreme_values)])

       Invoice StockCode                          Description  Quantity          InvoiceDate  Price  Customer_ID         Country
65091   495194     20993                 JAZZ HEARTS MEMO PAD      9312  2010-01-21 15:11:00   0.10      13902.0         DENMARK
90857   497946     37410   BLACK AND WHITE PAISLEY FLOWER MUG     19152  2010-02-15 11:57:00   0.10      13902.0         DENMARK
93677   498152     85220      SMALL FAIRY CAKE FRIDGE MAGNETS      9456  2010-02-17 10:51:00   0.30      13902.0         DENMARK
127166  501534     21099          SET/6 STRAWBERRY PAPER CUPS     12960  2010-03-17 13:09:00   0.10      13902.0         DENMARK
127167  501534     21092        SET/6 STRAWBERRY PAPER PLATES     12480  2010-03-17 13:09:00   0.10      13902.0         DENMARK
127168  501534     21091          SET/6 WOODLAND PAPER PLATES     12960  2010-03-17 13:09:00   0.10      13902.0         DENMARK
127169  501534     21085            SET/6 WOODLAND PAPER CUPS     12744  2010-03-17 13:09:00   0.

# Price Analysis

Rows with `Price = 0` are removed because they do not represent paid product purchases.

## Basic Information

In [356]:
print(f"Data type: {df["Price"].dtypes}")
print(f"Number of Missing Values: {df['Price'].isnull().sum()}")
print(f"Number of unique values: {df['Price'].nunique()}")

Data type: float64
Number of Missing Values: 0
Number of unique values: 308


## Statistics

In [357]:
print(df["Price"].describe())

count    406323.000000
mean          2.991565
std           4.285914
min           0.000000
25%           1.250000
50%           1.950000
75%           3.750000
max         295.000000
Name: Price, dtype: float64


## Price = 0.0

In [358]:
print(
    f"Number of times price appeared to be 0.0: {df[df["Price"] == 0].value_counts().sum()}\n"
)
print(f"{df[df["Price"] == 0].describe()}")

Number of times price appeared to be 0.0: 28

         Quantity  Price   Customer_ID
count   28.000000   28.0     28.000000
mean    30.678571    0.0  14078.142857
std    121.605024    0.0   1736.849931
min      1.000000    0.0  12417.000000
25%      1.000000    0.0  12647.000000
50%      4.000000    0.0  13321.500000
75%     10.500000    0.0  14752.000000
max    648.000000    0.0  18071.000000


In [359]:
# --------------- Data where price is 0 ---------------
print(df[df["Price"] == 0])

       Invoice StockCode                        Description  Quantity          InvoiceDate  Price  Customer_ID         Country
4674    489825     22076                   6 RIBBONS EMPIRE        12  2009-12-02 13:34:00    0.0      16126.0  UNITED KINGDOM
6781    489998     48185                DOOR MAT FAIRY CAKE         2  2009-12-03 11:19:00    0.0      15658.0  UNITED KINGDOM
18738   490961     22065      CHRISTMAS PUDDING TRINKET POT         1  2009-12-08 15:25:00    0.0      14108.0  UNITED KINGDOM
18739   490961     22142        CHRISTMAS CRAFT WHITE FAIRY        12  2009-12-08 15:25:00    0.0      14108.0  UNITED KINGDOM
32916   492079     85042          ANTIQUE LILY FAIRY LIGHTS         8  2009-12-15 13:49:00    0.0      15070.0  UNITED KINGDOM
40101   492760     21143     ANTIQUE GLASS HEART DECORATION        12  2009-12-18 14:22:00    0.0      18071.0  UNITED KINGDOM
47126   493761     79320                    FLAMINGO LIGHTS        24  2010-01-06 14:54:00    0.0      14258.0 

## Extreme Values

In [360]:
print(df["Price"].value_counts().sort_index().tail(10))

Price
65.00      4
69.95      3
70.00      1
79.95     53
110.00     5
125.00    14
129.95     1
145.00     6
165.00    10
295.00    32
Name: count, dtype: int64


## Cleaning

In [361]:
df = clean_prices(df)
print(
    f"Prices equivalent to 0 after cleaning: {df[df["Price"] == 0].value_counts().sum()}"
)

Prices equivalent to 0 after cleaning: 0


# Country Analysis

`Country` contains 37 unique values and has no missing values. The country values are already in a consistent format, so no cleaning is required.

## Basic Information

In [362]:
print(f"Data type: {df["Country"].dtypes}")
print(f"Number of unique countries: {df['Country'].nunique()}")
print(f"Number of missing values: {df['Country'].isnull().sum()}")

Data type: str
Number of unique countries: 37
Number of missing values: 0


## Value Counts

In [363]:
print(df["Country"].value_counts().sort_values(ascending=False))

Country
UNITED KINGDOM          370455
EIRE                      8388
GERMANY                   7376
FRANCE                    5288
NETHERLANDS               2661
SPAIN                     1195
SWITZERLAND               1146
BELGIUM                    988
PORTUGAL                   956
SWEDEN                     841
CHANNEL ISLANDS            817
ITALY                      695
AUSTRALIA                  625
CYPRUS                     539
GREECE                     511
AUSTRIA                    504
DENMARK                    408
NORWAY                     362
FINLAND                    338
UNITED ARAB EMIRATES       313
UNSPECIFIED                277
USA                        225
POLAND                     181
MALTA                      168
JAPAN                      164
LITHUANIA                  154
SINGAPORE                  117
CANADA                      77
THAILAND                    76
ISRAEL                      74
ICELAND                     71
RSA                         65


# InvoiceDate Analysis

`InvoiceDate` is stored as a string, so it is converted to a datetime data type for proper date and time operations.

## Basic Information

In [364]:
print(f"Data type: {df["InvoiceDate"].dtypes}")
print(f"Number of Missing Values: {df['InvoiceDate'].isnull().sum()}")
print(f"Number of unique values: {df['InvoiceDate'].nunique()}")

Data type: str
Number of Missing Values: 0
Number of unique values: 17809


## Sample Values

In [365]:
print(df["InvoiceDate"].sample(10))

436529    2010-11-05 12:38:00
228558    2010-06-08 14:44:00
64975     2010-01-21 14:30:00
307526    2010-08-16 16:54:00
74911     2010-01-29 14:37:00
11932     2009-12-04 16:37:00
353241    2010-09-23 16:57:00
178670    2010-04-29 10:54:00
209522    2010-05-24 15:36:00
254670    2010-06-29 14:14:00
Name: InvoiceDate, dtype: str


## Cleaning

In [368]:
df = clean_invoice_date(df)
print(f"Data type of Invoice Date after cleaning: {df['InvoiceDate'].dtypes}")

Data type of Invoice Date after cleaning: datetime64[us]


## Ranges

In [367]:
print(f"Minimum date: {df['InvoiceDate'].min()}")
print(f"Maximum date: {df['InvoiceDate'].max()}")
print(f"Range: {df['InvoiceDate'].max() - df['InvoiceDate'].min()}")

Minimum date: 2009-12-01 07:45:00
Maximum date: 2010-12-09 20:01:00
Range: 373 days 12:16:00
